# Analyzing GitHub Actions Usage Across Organization Repositories

A reference notebook for auditing Actions consumption across every repository in one organization: inventory the repos, collect workflow runs per repo, then summarize run volume, failure rate, and repeat-failure (flaky) jobs from the same dataset.

## Purpose

Organizations with dozens of repositories lose track of where Actions time goes: which repos run the most jobs, which workflows fail most often, and which jobs fail intermittently and get re-run. This notebook defines a repeatable three-step audit (inventory, collect, summarize) that turns per-repo run listings into one organization-wide table.

## When to use

| Situation | Why this audit helps |
| --- | --- |
| Runner budget review | Ranks repos and workflows by run count so the largest consumers are visible first |
| Reliability review | Surfaces workflows with the highest failure rate and jobs that fail then pass on retry |
| Cleanup pass | Finds repos with no runs in the window, which are candidates for workflow removal or archival |

## Prerequisites

- The `gh` CLI installed and authenticated with read access to the organization (`gh auth status` reports a logged-in account).
- Organization member or owner role sufficient to list repositories and read Actions runs.
- Python with only the standard library (`subprocess`, `json`, `shutil`, `collections`, `datetime`); every cell below falls back to embedded sample data when the CLI or network is unavailable.

## Steps

The three cells below implement the audit. Each live-collection cell tries the `gh` CLI first and uses the embedded sample payload only when the CLI is missing or returns no rows, so the notebook runs end to end in a sandbox and against a real organization without edits.

### 1. Inventory repositories in the organization

List every non-archived repository. The live path pages through the repo list; the fallback keeps three representative repos so downstream cells still execute.

In [ ]:
# last_verified: 2026-10-11 · GitHub n/a
import json
import shutil
import subprocess

ORG = "my-org"
LIMIT = 100

SAMPLE_REPOS = [{"name": "web-app"}, {"name": "api-service"}, {"name": "infra-modules"}]


def list_org_repos(org, limit=100):
    """Return [{"name": ...}] via the gh CLI, or the sample list when unavailable."""
    gh = shutil.which("gh")
    if gh is None:
        print("gh CLI not found; using sample repository list.")
        return list(SAMPLE_REPOS)
    try:
        proc = subprocess.run(
            [gh, "repo", "list", org, "--limit", str(limit), "--json", "name"],
            capture_output=True, text=True, timeout=60, check=False,
        )
    except (subprocess.SubprocessError, OSError) as exc:
        print(f"repo listing failed ({exc}); using sample repository list.")
        return list(SAMPLE_REPOS)
    if proc.returncode != 0 or not proc.stdout.strip():
        print(f"repo listing returned no rows (exit {proc.returncode}); using sample list.")
        return list(SAMPLE_REPOS)
    try:
        rows = json.loads(proc.stdout)
    except json.JSONDecodeError:
        print("repo listing was not valid JSON; using sample repository list.")
        return list(SAMPLE_REPOS)
    repos = [{"name": r.get("name", "")} for r in rows if r.get("name")]
    return repos or list(SAMPLE_REPOS)


repos = list_org_repos(ORG, LIMIT)
print(f"repositories in scope: {len(repos)}")
for r in repos[:10]:
    print(f"- {r['name']}")


### 2. Collect workflow runs per repository

Pull the recent runs for each repository in scope. The live path requests the run id, workflow name, conclusion, and creation timestamp; the fallback reuses one fixed payload so the summary math stays deterministic.

In [ ]:
# last_verified: 2026-10-11 · GitHub n/a
import json
import shutil
import subprocess

SAMPLE_RUNS = [
    {"repo": "web-app", "workflow": "ci", "conclusion": "success", "createdAt": "2026-10-01"},
    {"repo": "web-app", "workflow": "ci", "conclusion": "failure", "createdAt": "2026-10-02"},
    {"repo": "web-app", "workflow": "ci", "conclusion": "success", "createdAt": "2026-10-03"},
    {"repo": "api-service", "workflow": "ci", "conclusion": "failure", "createdAt": "2026-10-01"},
    {"repo": "api-service", "workflow": "release", "conclusion": "success", "createdAt": "2026-10-02"},
    {"repo": "infra-modules", "workflow": "ci", "conclusion": "success", "createdAt": "2026-10-02"},
]


def collect_runs(org, repos, limit=50):
    """Return run dicts via the gh CLI, or SAMPLE_RUNS when unavailable."""
    gh = shutil.which("gh")
    if gh is None:
        print("gh CLI not found; using sample run payload.")
        return list(SAMPLE_RUNS)
    collected = []
    for entry in repos:
        name = entry["name"]
        try:
            proc = subprocess.run(
                [gh, "run", "list", "--repo", f"{org}/{name}",
                 "--limit", str(limit), "--json", "name,conclusion,createdAt"],
                capture_output=True, text=True, timeout=60, check=False,
            )
        except (subprocess.SubprocessError, OSError):
            continue
        if proc.returncode != 0 or not proc.stdout.strip():
            continue
        try:
            rows = json.loads(proc.stdout)
        except json.JSONDecodeError:
            continue
        for row in rows:
            collected.append({
                "repo": name,
                "workflow": row.get("name", "unknown"),
                "conclusion": row.get("conclusion", "unknown"),
                "createdAt": (row.get("createdAt", "") or "")[:10],
            })
    if not collected:
        print("no live runs collected; using sample run payload.")
        return list(SAMPLE_RUNS)
    return collected


runs = collect_runs(ORG, repos)
print(f"runs collected: {len(runs)}")
for row in runs[:6]:
    print(row)


### 3. Summarize volume, failure rate, and flaky jobs

Aggregate the run list into per-workflow totals: run count, failure count, failure rate, and a repeat-failure flag for workflows that show both a failure and a success in the window (the simplest flaky-job signal available from conclusions alone).

In [ ]:
# last_verified: 2026-10-11 · GitHub n/a
from collections import defaultdict


def summarize_runs(run_rows):
    """Aggregate runs to {(repo, workflow): stats} with failure rate and flaky flag."""
    buckets = defaultdict(lambda: {"runs": 0, "failures": 0, "saw_success": False, "saw_failure": False})
    for row in run_rows:
        key = (row.get("repo", "unknown"), row.get("workflow", "unknown"))
        slot = buckets[key]
        slot["runs"] += 1
        conclusion = (row.get("conclusion") or "unknown").lower()
        if conclusion == "failure":
            slot["failures"] += 1
            slot["saw_failure"] = True
        elif conclusion == "success":
            slot["saw_success"] = True
    table = []
    for (repo, workflow), slot in sorted(buckets.items()):
        rate = slot["failures"] / slot["runs"] if slot["runs"] else 0.0
        table.append({
            "repo": repo,
            "workflow": workflow,
            "runs": slot["runs"],
            "failures": slot["failures"],
            "failure_rate": round(rate, 3),
            "flaky": bool(slot["saw_success"] and slot["saw_failure"]),
        })
    return sorted(table, key=lambda r: (-r["runs"], -r["failure_rate"]))


summary = summarize_runs(runs)
print(f"{'repo':<15}{'workflow':<10}{'runs':>6}{'fail':>6}{'rate':>7}  flaky")
for row in summary:
    print(f"{row['repo']:<15}{row['workflow']:<10}{row['runs']:>6}{row['failures']:>6}{row['failure_rate']:>7}  {row['flaky']}")


## Verify

The checks below confirm the summary math on the fixed sample payload: totals per workflow, failure-rate rounding, sort order, and the repeat-failure flag.

In [ ]:
# last_verified: 2026-10-11 · GitHub n/a
table = {(r["repo"], r["workflow"]): r for r in summarize_runs(SAMPLE_RUNS)}

assert table[("web-app", "ci")]["runs"] == 3
assert table[("web-app", "ci")]["failures"] == 1
assert table[("web-app", "ci")]["failure_rate"] == 0.333
assert table[("web-app", "ci")]["flaky"] is True
assert table[("api-service", "ci")]["flaky"] is False
assert table[("infra-modules", "ci")]["failures"] == 0
assert summarize_runs([]) == []

print("verify: 7 assertions passed")


## Common errors

- Empty result with exit code 0: the token lacks organization read scope or the org name is misspelled; confirm with `gh auth status` and re-check the org spelling before widening the run limit.
- Rate limiting on large organizations: collecting per-repo runs issues one CLI call per repository; lower the per-repo limit or restrict the repo list when the run stalls partway.
- Misleading zero-failure tables: conclusions other than `success` or `failure` (cancelled, skipped, timed out) are counted as runs but not failures; filter those conclusions explicitly when a strict pass/fail ratio is required.

## References

- GitHub CLI manual: repository listing and workflow-run listing commands
- Actions usage and billing concepts: included minutes, per-repository breakdowns, and artifact retention
- Workflow observability practices: failure-rate dashboards and re-run analysis for flaky jobs
